<a href="https://colab.research.google.com/github/ThanhB18059162022/MMRec/blob/dev/preprocessing/1splitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## preprocessing/
## Tiền xử lý bước 1
Bước này tạo file sports14-indexed-v4.inter. File này chia dữ liệu theo Cách làm mới (Per-user Split)

# 基于rating2inter.ipynb生成的5-core交互图，Train/Validation/Test data splitting
- Based on generated interactions, perform data splitting


In [1]:
import os, csv
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
PATH = "/content/drive/MyDrive/Colab Notebooks/CTH001/MMRec/"

In [4]:
!mkdir "{PATH}/data/1splitting"

In [5]:
!cp "{PATH}/data/0rating2inter/sports14-indexed.inter" "."

## 直接加载现成的, Load interactions

In [6]:
rslt_file = 'sports14-indexed.inter'
df = pd.read_csv(rslt_file, sep='\t')
print(f'shape: {df.shape}')
df[:4]

shape: (296337, 5)


,userID,itemID,rating,timestamp,x_label
0,0,0,5.0,1390694400,0
1,1,0,5.0,1328140800,0
2,2,0,4.0,1330387200,0
3,3,0,4.0,1328400000,0


In [ ]:
import random
import numpy as np

Đoạn code này thực hiện hai thao tác xử lý dữ liệu có vẻ trái ngược nhau nhưng lại rất phổ biến trong quá trình chuẩn bị dữ liệu cho hệ thống gợi ý.

In [41]:
# 1. Xáo trộn dữ liệu (Shuffle)
# df = df.sample(frac=1).reset_index(drop=True)
# df.sample(frac=1): Lấy mẫu ngẫu nhiên 100% dữ liệu từ DataFrame. Điều này tương đương với việc trộn bài (xáo trộn thứ tự các dòng).
# reset_index(drop=True): Sau khi xáo trộn, các chỉ số (index) cũ sẽ bị đảo lộn. Lệnh này đặt lại index mới từ 0 đến $N-1$ và xóa bỏ cột index cũ.
# Tại sao làm vậy? Để đảm bảo tính ngẫu nhiên, loại bỏ mọi sự sắp xếp vô tình có sẵn trong dữ liệu gốc (ví dụ: dữ liệu có thể đã được gom nhóm theo thời gian hoặc theo shop trước đó), giúp các bước xử lý sau khách quan hơn.


# Con số 42 là con số may mắn huyền thoại trong giới Data Science,
# bạn có thể chọn số bất kỳ (1, 100, 2024...)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# 2. Sắp xếp theo ID người dùng
# df.sort_values(by=['userID'], inplace=True)
# df[:20]

# Tại sao làm vậy? Trong mô hình GCN hoặc các hệ thống gợi ý, việc sắp xếp theo userID cực kỳ quan trọng vì:
# Quản lý tương tác: Giúp bạn dễ dàng thấy được toàn bộ lịch sử các sản phẩm mà một người dùng đã tương tác (tất cả các dòng của User 0 nằm cạnh nhau, sau đó đến User 1, v.v.).
# Tối ưu tính toán: Khi nạp dữ liệu vào mô hình, việc các tương tác của cùng một người dùng nằm liên tiếp giúp quá trình xử lý theo lô (batch processing) hoặc trích xuất hàng xóm trên đồ thị diễn ra hiệu quả hơn.

# 2. Sắp xếp theo userID, NẾU trùng userID thì sắp xếp theo timestamp (hoặc sản phẩm)
# Việc này đảm bảo thứ tự bên trong mỗi User luôn cố định
df.sort_values(by=['userID', 'timestamp'], inplace=True)

df[:20]

,userID,itemID,rating,timestamp,x_label
13178,0,11981,2.0,1390694400,0
130316,0,0,5.0,1390694400,0
245498,0,15852,5.0,1390694400,0
79048,0,3327,3.0,1391990400,1
181775,0,13372,5.0,1391990400,1
284826,0,17787,3.0,1391990400,1
40213,0,5458,5.0,1405123200,2
229661,0,3369,5.0,1405123200,2
209165,1,7215,5.0,1285372800,0
135185,1,3087,5.0,1293494400,0


Đoạn code này có nhiệm vụ chuyển đổi dữ liệu từ dạng bảng (DataFrame) sang dạng Từ điển danh sách kề (Adjacency List/Dictionary).

Trong các hệ thống gợi ý và đặc biệt là mô hình đồ thị như SMORE, đây là bước chuẩn bị để mô hình biết được mỗi người dùng đã tương tác với những sản phẩm nào một cách nhanh nhất.

In [42]:
# 1. Khai báo tên cột
uid_field, iid_field = 'userID', 'itemID'

# 2. Nhóm dữ liệu (Groupby)
uid_freq = df.groupby(uid_field)[iid_field]
u_i_dict = {}
for u, u_ls in uid_freq:
    u_i_dict[u] = list(u_ls)
list(u_i_dict.items())[:3]

[(0, [11981, 0, 15852, 3327, 13372, 17787, 5458, 3369]),
 (1,
  [7215,
   3087,
   7169,
   1542,
   11502,
   6677,
   9198,
   4281,
   13468,
   0,
   2322,
   5044,
   15278,
   6554,
   4123,
   14212,
   8802,
   15249,
   2374]),
 (2,
  [10218,
   6298,
   7950,
   14657,
   0,
   3011,
   14242,
   8776,
   1114,
   2950,
   10837,
   500,
   11360,
   14699,
   9254,
   6445,
   4155,
   15075,
   14212,
   9841,
   3661,
   15360,
   9250])]

Tại sao bước này quan trọng đối với SMORE-GCN?
Trong luận văn của bạn, cấu trúc dữ liệu này cực kỳ hữu ích vì:

Xây dựng đồ thị (Graph Construction): Mô hình GCN cần biết các "hàng xóm" của một Node. Từ điển này chính là danh sách hàng xóm của các Node người dùng.

Lấy mẫu tiêu cực (Negative Sampling): Khi huấn luyện, mô hình cần chọn ra những sản phẩm mà người dùng chưa từng mua. Dựa vào u_i_dict[u], máy tính sẽ biết sản phẩm nào đã mua rồi để tránh chọn trùng.

Tốc độ: Truy xuất thông tin từ một từ điển (dict) trong Python nhanh hơn rất nhiều so với việc lọc (filter) trên một DataFrame khổng lồ mỗi khi cần biết "User này thích cái gì?".

In [43]:
list(u_i_dict.keys())[:3]

[0, 1, 2]

Đây là đoạn code thực hiện việc Phân chia dữ liệu (Splitting) cho từng người dùng cụ thể. Tuy nhiên, cách làm này khác với cách chia theo thời gian (Temporal Split) ở trên; đây là cách chia theo tỷ lệ trên từng User (Per-user Splitting).

Mục tiêu của nó là tạo ra một danh sách các nhãn (0 cho Train, 1 cho Val, 2 cho Test) sao cho mỗi người dùng đều có ít nhất một vài sản phẩm trong cả 3 tập dữ liệu.

In [45]:
new_label = []
# 1. Sắp xếp danh sách User
u_ids_sorted = sorted(u_i_dict.keys())

# 2. Vòng lặp xử lý từng User
# Code chia làm 2 trường hợp dựa trên số lượng tương tác (n_items) của mỗi User:

# Trường hợp A: User có ít tương tác (dưới 10 món)
# Nếu User chỉ mua 5 món, việc chia 80% có thể làm mất tập Val hoặc Test.
# Giải pháp: Lấy đúng 1 món cho Test (nhãn 2), 1 món cho Val (nhãn 1), và tất cả phần còn lại cho Train (nhãn 0).
# Ví dụ: User có 5 món => Nhãn sẽ là [0, 0, 0, 1, 2].

# Trường hợp B: User có nhiều tương tác (từ 10 món trở lên)
# Đây là cách chia 80/10/10 truyền thống nhưng áp dụng trên từng người dùng.
# Ví dụ: User có 20 món => train_len=16, val_len=2, test_len=2. Nhãn sẽ là 16 số 0, 2 số 1 và 2 số 2.
for u in u_ids_sorted:
    items = u_i_dict[u]
    # get num interact
    n_items = len(items)
    if n_items < 10:
        # take 1 for test 1 for val and rest for train
        tmp_ls = [0] * (n_items - 2) + [1] + [2]
    else:
        # split 80% train, 10% val, 10% test
        val_test_len = int(n_items * 0.2)
        train_len = n_items - val_test_len
        val_len = val_test_len // 2
        test_len = val_test_len - val_len
        tmp_ls = [0] * train_len + [1] * val_len + [2] * test_len
    new_label.extend(tmp_ls)

new_label[:50]

[0,
 0,
 0,
 0,
 0,
 0,
 1,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 2,
 2,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 2,
 2]

Sau khi tạo xong nhãn cho một User, nó "nối" danh sách đó vào danh sách tổng new_label. Cuối cùng, new_label sẽ có độ dài bằng đúng tổng số dòng trong DataFrame của bạn

Sự khác biệt lớn nhất giữa nhãn mới này và nhãn bạn đã làm ở các bước trước nằm ở **Phương pháp chia (Splitting Strategy)**.

Dưới đây là bảng so sánh để bạn dễ hình dung cho luận văn:

### 1. So sánh hai loại nhãn

| Đặc điểm | Cách làm cũ (Global Split) | Cách làm mới (Per-user Split) |
| --- | --- | --- |
| **Cơ chế** | Cắt một nhát tại mốc 80% của **toàn bộ file**. | Chia 80/10/10 trên **từng người dùng một**. |
| **Tính công bằng** | Một số User mua đồ gần đây sẽ bị đẩy hết vào tập Test. Những User mua đồ quá cũ sẽ bị đẩy hết vào tập Train. | **Mọi User** đều xuất hiện trong tập Train, tập Val và tập Test. |
| **Lợi ích** | Phản ánh đúng dòng thời gian thực tế của cả hệ thống. | Giúp mô hình học được "gu" của tất cả người dùng trước khi yêu cầu nó dự đoán (Test). |
| **Rủi ro** | Có những User "mất tích" hoàn toàn trong tập Train (vì họ chỉ mới mua đồ gần đây). | Nếu dữ liệu không sắp xếp theo `timestamp` bên trong mỗi User, việc chia này sẽ mất ý nghĩa thực tế. |

---

### 2. Tại sao lại cần nhãn mới này cho mô hình GCN/SMORE?

Trong các mô hình đồ thị (GCN), việc đảm bảo **mỗi Node User đều có ít nhất một vài cạnh (edges) trong tập Train** là rất quan trọng.

* **Nếu dùng cách cũ:** User A có thể không có một tương tác nào trong tập Train (vì toàn bộ 5 món đồ của họ đều nằm trong 10% dữ liệu mới nhất của hệ thống). Khi đó, Node User A sẽ là "Node mồ côi" trong lúc huấn luyện $\rightarrow$ Mô hình không học được Embedding cho User A.
* **Với cách mới:** Bạn ép buộc User A phải đóng góp ít nhất 3 món vào tập Train để mô hình học được sở thích của họ, sau đó mới dùng 2 món còn lại để Test. Điều này giúp đánh giá khả năng "Gợi ý cá nhân hóa" chính xác hơn.

---

### 3. Phân tích ví dụ bạn đưa ra

```text
userID  itemID  rating  timestamp   x_label
0       0       5.0     1390694400  0
0       1       5.0     1328140800  0
0       2       4.0     1330387200  0
0       3       4.0     1328400000  0

```

Ở ví dụ này, 4 dòng đầu của `userID: 0` đều có nhãn `0` (Train).

* Nếu User này có tổng cộng 6 tương tác, thì 2 dòng tiếp theo của họ sẽ lần lượt mang nhãn `1` (Val) và `2` (Test).
* **Lưu ý quan trọng:** Hãy nhìn vào cột `timestamp` trong ví dụ của bạn:
* Dòng 1: `1390694400` (Năm 2014)
* Dòng 2: `1328140800` (Năm 2012)
* Dòng 3: `1330387200` (Năm 2012)
* Dòng 4: `1328400000` (Năm 2012)



> **Nguy hiểm:** Dòng đầu tiên (2014) lại đang đứng trước các dòng năm 2012. Nếu bạn gán nhãn `0` (Train) cho dòng 2014 và nhãn `2` (Test) cho dòng 2012, bạn đang dùng **tương lai để dự đoán quá khứ**.

**Lời khuyên:** Bạn cần chạy lệnh `df.sort_values(by=['userID', 'timestamp'], inplace=True)` **TRƯỚC** khi tạo `u_i_dict` và `new_label` để đảm bảo món đồ cuối cùng của mỗi User (nhãn 2) luôn là món đồ mới nhất họ mua nhé!

**Bạn có muốn mình hướng dẫn cách gán `new_label` này vào DataFrame sao cho chuẩn xác nhất không?**

In [47]:
df['x_old_label'] = df['x_label'].copy()

In [55]:
# df.drop(columns=['x_label1'], inplace=True)

In [56]:
df['x_label'] = new_label
df[:20]

,userID,itemID,rating,timestamp,x_label,x_old_label
13178,0,11981,2.0,1390694400,0,0
130316,0,0,5.0,1390694400,0,0
245498,0,15852,5.0,1390694400,0,0
79048,0,3327,3.0,1391990400,0,1
181775,0,13372,5.0,1391990400,0,1
284826,0,17787,3.0,1391990400,0,1
40213,0,5458,5.0,1405123200,1,2
229661,0,3369,5.0,1405123200,2,2
209165,1,7215,5.0,1285372800,0,0
135185,1,3087,5.0,1293494400,0,0


In [57]:
rslt_file[:-6]

'sports14-indexed'

In [58]:
new_labeled_file = rslt_file[:-6] + '-v4.inter'
df.to_csv(os.path.join('./', new_labeled_file), sep='\t', index=False)
print('done!!!')

done!!!


In [59]:
!cp "{new_labeled_file}" "{PATH}/data/1splitting/{new_labeled_file}"

## Reload

In [60]:
indexed_df = pd.read_csv(new_labeled_file, sep='\t')
print(f'shape: {indexed_df.shape}')
indexed_df[:20]

shape: (296337, 6)


,userID,itemID,rating,timestamp,x_label,x_old_label
0,0,11981,2.0,1390694400,0,0
1,0,0,5.0,1390694400,0,0
2,0,15852,5.0,1390694400,0,0
3,0,3327,3.0,1391990400,0,1
4,0,13372,5.0,1391990400,0,1
5,0,17787,3.0,1391990400,0,1
6,0,5458,5.0,1405123200,1,2
7,0,3369,5.0,1405123200,2,2
8,1,7215,5.0,1285372800,0,0
9,1,3087,5.0,1293494400,0,0


In [ ]:
u_id_str, i_id_str = 'userID', 'itemID'
u_uni = indexed_df[u_id_str].unique()
c_uni = indexed_df[i_id_str].unique()

print(f'# of unique learners: {len(u_uni)}')
print(f'# of unique courses: {len(c_uni)}')

print('min/max of unique learners: {0}/{1}'.format(min(u_uni), max(u_uni)))
print('min/max of unique courses: {0}/{1}'.format(min(c_uni), max(c_uni)))


# of unique learners: 35598
# of unique courses: 18357
min/max of unique learners: 0/35597
min/max of unique courses: 0/18356


In [61]:
u_id_str, i_id_str = 'userID', 'itemID'
u_uni = indexed_df[u_id_str].unique()
c_uni = indexed_df[i_id_str].unique()

print(f'# of unique learners: {len(u_uni)}')
print(f'# of unique courses: {len(c_uni)}')

print('min/max of unique learners: {0}/{1}'.format(min(u_uni), max(u_uni)))
print('min/max of unique courses: {0}/{1}'.format(min(c_uni), max(c_uni)))


# of unique learners: 35598
# of unique courses: 18357
min/max of unique learners: 0/35597
min/max of unique courses: 0/18356
